# CSIゾーン分類：深層学習（MLP / 1D-CNN / 2D-CNN）を Google Colab の GPU で学習する

手順の詳細は `docs/colab.md`。上のセルから順に実行する。

**事前準備（PC側）**
1. `server/` で `python -m tools.pack_for_colab` を実行し、`data/colab_upload.zip` を作る
2. Google Drive の **マイドライブ/csi-location/** に `colab_upload.zip` をアップロードする

**このノートブックの準備**
- メニュー「ランタイム → ランタイムのタイプを変更」で **T4 GPU** を選ぶ

**途中で切断されたら**: ランタイムに再接続して、上からもう一度実行する。比較（手順4）は記録済みの回をとばして続きから回る。
結果はすべて **マイドライブ/csi-location/results/** に保存される。

## 1. GPUの確認
`Tesla T4` などと表示されればOK。`command not found` ならGPUが選ばれていない（上の「事前準備」を参照）。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive をつなぎ、コードとデータを展開する
許可を求める画面が出たら、自分のGoogleアカウントで許可する。展開先はColabのローカルディスク（`/content/work`、速い）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
DRIVE_DIR = '/content/drive/MyDrive/csi-location'   # zipを置いたフォルダ
ZIP_PATH = os.path.join(DRIVE_DIR, 'colab_upload.zip')
WORK_DIR = '/content/work'
RESULTS = os.path.join(DRIVE_DIR, 'results')         # 結果の保存先（Drive。切断されても残る）
os.makedirs(RESULTS, exist_ok=True)

assert os.path.exists(ZIP_PATH), f'{ZIP_PATH} がありません。PCで作ったzipをアップロードしてください'
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(WORK_DIR)
os.chdir(os.path.join(WORK_DIR, 'server'))
print('作業フォルダ:', os.getcwd())
print('データ:', {f: f'{os.path.getsize("data/" + f) / 1e6:.0f}MB' for f in os.listdir('data')})

## 3. 環境の確認とテスト
PyTorch はColabに最初から入っている。テストが `OK` で終わればコードが動く。

In [ ]:
import sys, torch, numpy, sklearn
print('python', sys.version.split()[0], '/ torch', torch.__version__, '/ numpy', numpy.__version__, '/ scikit-learn', sklearn.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし（CPUで動きます。かなり遅い）')
!python -m unittest discover -s tests -t . 2>&1 | tail -3

## 4. 比較：MLP / 1D-CNN / 2D-CNN × 窓の長さ × シード

- 評価はPCの RandomForest と同じ「1周抜き」（`--split round`）。
- 結果は終わった回から `results/compare_deep.csv` に追記される。**切断されても、このセルをもう一度実行すれば続きから回る**（`--resume`）。
- 最初からやり直したいときは、Driveの `results/compare_deep.csv` を消してから実行する。
- GPUは1つなので `--jobs 1`。

In [ ]:
MODELS = 'mlp,cnn1d,cnn2d'
WINDOWS = '20,40,60'      # 20 = 1秒、40 = 2秒、60 = 3秒
SEEDS = '42,43,44'
OUT = os.path.join(RESULTS, 'compare_deep')

!python -m training.compare --data data/calib.jsonl --baseline data/baseline.jsonl     --models {MODELS} --windows {WINDOWS} --seeds {SEEDS} --split round --stride 2     --device cuda --jobs 1 --resume --out "{OUT}" --cache-dir /content/cache 2>&1 | tee -a "{OUT}.log" | grep -v "^$" 

## 5. 比較表を見る
PCの基準: RandomForest（窓1秒）正解率 0.601 / macro-F1 0.580 / empty再現率 0.884。

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(OUT + '.md', encoding='utf-8').read()))

## 6. 本番用のモデルを学習して保存する

比較（手順4）は評価だけでモデルを保存しない。リアルタイム推定で使うモデルはここで作る。
`FINAL` を手順5で良かった組み合わせに書き換えて実行する。保存先は `results/model_<モデル>_w<窓>.pkl`（すでにあればとばす）。

In [ ]:
FINAL = [('mlp', 20), ('cnn1d', 60), ('cnn2d', 60)]   # (モデル, 窓) を手順5の結果に合わせて変える

for algo, window in FINAL:
    path = os.path.join(RESULTS, f'model_{algo}_w{window}.pkl')
    if os.path.exists(path):
        print('とばす（もうある）:', path)
        continue
    print('学習:', algo, window)
    !python -m training.train --data data/calib.jsonl --baseline data/baseline.jsonl         --algo {algo} --window {window} --split round --stride 2 --device cuda         --model-out "{path}" 2>&1 | grep "evaluation:\|model saved\|Error" 

## 7. 保存したモデルがCPU（PC）で読めるか確認する
`cpu` と表示されればOK。PCの `server/data/` にダウンロードして、`localization.estimate --model data/model_....pkl` で使う。

In [ ]:
import glob, numpy as np
from models.registry import load_bundle
from features import input_shape
for path in sorted(glob.glob(os.path.join(RESULTS, 'model_*.pkl'))):
    b = load_bundle(path)
    m = b['model']
    x = np.zeros((1, *input_shape(b['input_kind'], len(b['sensor_ids']), b['expected_len'], b['window_frames'])), dtype=np.float32)
    m.prepare_realtime()
    p = m.predict_proba(x)
    print(os.path.basename(path), '| 評価 正解率', round(b['metrics']['accuracy'], 3), '| 推論', next(m.net.parameters()).device, p.shape)